In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib, json, zipfile, shutil, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, sosfiltfilt
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from threadpoolctl import threadpool_limits
try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/obp_utils.py').exists()), Path.cwd())
RAW = ROOT / 'data/raw/hitting'
OUT = ROOT / 'results/lower_body_force_proxies_v3'
FIG = ROOT / 'figures/lower_body_force_proxies_v3'
for p in (RAW, OUT, FIG):
    p.mkdir(parents=True, exist_ok=True)
AUTO_DOWNLOAD = True
SEED = 20261024
PRIMARY_WINDOW_S = .150
END_BEFORE_CONTACT_S = .010
FILTER_HZ = 12.
MIN_PROFILE_SWINGS = 5
OUTER_FOLDS = 5
INNER_FOLDS = 4
CV_REPEATS = 2
ALPHAS = [0.1, 1., 10., 100., 1000.]
BOOTSTRAPS = 2000
CLUSTER_BOOTSTRAPS = 100
RUN_MODEL_SENSITIVITY = True
G = 9.80665
threadpool_limits(limits=1)
plt.style.use('seaborn-v0_8-whitegrid')
INTRO = '''Study 18 — Lower-Body Movement Strategies and Ground Reaction Force Estimation

Question: do lower-body motion and modeled center-of-mass movement estimate selected
ground reaction force measures in hitters not used for model training?
Primary measurements use contact minus 150 ms through contact minus 10 ms. Contact
is an observed alignment event; this is retrospective assessment, not live forecasting.
No force-derived landing timestamp enters primary features or primary eligibility.
Official-plant features form an explicitly force-assisted benchmark only.
The published centerofmass trajectory is a model-based COM estimate, not a hip midpoint.
COM features are compared against a simpler pelvis/joint model. COM is not independently
validated here, and differentiating its trajectory amplifies tracking/filtering error.
Kinematic landing is an exploratory ankle-settling detector, not a validated foot-strike event.
All associations are observational; clusters are descriptive, not mechanical grades.
The source collection used optical motion capture. Success here would not validate phone video.
Raw data are downloaded locally, never embedded in this notebook.
The study includes a retained-trial measurement audit, explicitly labeled exclusion sensitivities,
and a same-fold comparison against uncalibrated and training-calibrated COM physics.
It compares COM-only and full models for lead vertical-force peak and braking impulse.
The original primary cohort and models remain unchanged. No timestamp is repaired.
'''
print(INTRO)
print('Run all cells in order. Dependencies: numpy pandas scipy matplotlib scikit-learn threadpoolctl ipython.')
print('First run downloads about 168 MB of archives plus small metadata/POI files; unpacked data need more space.')
print('Source: https://github.com/drivelineresearch/openbiomechanics/tree/dataset-v1/baseball_hitting')
print('Review source data license before use: https://github.com/drivelineresearch/openbiomechanics/blob/main/LICENSE-DATA.md')


In [ ]:
CSV_BASE = 'https://raw.githubusercontent.com/drivelineresearch/openbiomechanics/dataset-v1/baseball_hitting/data/'
RELEASE_BASE = 'https://github.com/drivelineresearch/openbiomechanics/releases/download/dataset-v1/'
ARCHIVE_HASHES = {
    'force_plate': 'b252e3c91cd6e668fd078af898a1a086bc95bd05352ed42debefaa226276aac1',
    'joint_angles': '98a405c5c4a27c1a599d3e30b39a3087180cc4d4bb69faf62204a1c06881be95',
    'landmarks': '9cc2bdc94f5f5a6c5d540f4abe459caae2c5164cd53f6a6fd23f4b2b7283bcb3'
}
EVENTS = ['fp_10_time', 'fp_100_time', 'contact_time']
JOINTS = ['lead_knee_angle_x', 'rear_knee_angle_x', 'lead_hip_angle_x', 'rear_hip_angle_x',
          'pelvis_angle_z', 'torso_angle_z']
LANDMARK_BASES = ['left_hip', 'right_hip', 'lajc', 'rajc', 'centerofmass']
LM_COLS = [f'{b}_{a}' for b in LANDMARK_BASES for a in 'xyz']
FORCE_COLS = [f'{leg}_force_{a}' for leg in ['lead', 'rear'] for a in 'xyz']

def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def download(url, path):
    if not AUTO_DOWNLOAD:
        raise FileNotFoundError(f'Missing {path}. Supply the official input or enable AUTO_DOWNLOAD.')
    print('Downloading', path.name, flush=True)
    tmp = path.with_suffix(path.suffix + '.part')
    with urlopen(url, timeout=120) as src, tmp.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    tmp.replace(path)

for name, expected in ARCHIVE_HASHES.items():
    csv = RAW / f'{name}.csv'
    if not csv.exists():
        archive = RAW / f'hitting_{name}.zip'
        if not archive.exists():
            download(RELEASE_BASE + archive.name, archive)
        if file_hash(archive) != expected:
            raise ValueError(f'Archive checksum mismatch: {archive.name}')
        with zipfile.ZipFile(archive) as z:
            names = [n for n in z.namelist() if Path(n).name == csv.name]
            if len(names) != 1:
                raise ValueError(f'Expected one {csv.name} in archive.')
            tmp = csv.with_suffix('.csv.part')
            with z.open(names[0]) as src, tmp.open('wb') as dst:
                shutil.copyfileobj(src, dst)
            tmp.replace(csv)
for filename, remote in [('metadata.csv', 'metadata.csv'), ('poi_metrics.csv', 'poi/poi_metrics.csv')]:
    if not (RAW / filename).exists():
        download(CSV_BASE + remote, RAW / filename)

def load_signal(name, columns):
    frame = pd.read_csv(RAW / f'{name}.csv', usecols=['session_swing', 'time', *EVENTS, *columns],
                        dtype={'session_swing': str})
    if frame.session_swing.isna().any() or frame.duplicated(['session_swing', 'time']).any():
        raise ValueError(f'Missing trial ID or duplicate trial/time in {name}.')
    return {s: g.sort_values('time').reset_index(drop=True)
            for s, g in frame.groupby('session_swing', sort=False)}

angles = load_signal('joint_angles', JOINTS)
landmarks = load_signal('landmarks', LM_COLS)
forces = load_signal('force_plate', FORCE_COLS)
metadata = pd.read_csv(RAW / 'metadata.csv', dtype={'session_swing': str, 'user': str, 'session': str})
poi = pd.read_csv(RAW / 'poi_metrics.csv', dtype={'session_swing': str})
for name, frame in [('metadata', metadata), ('POI', poi)]:
    if frame.session_swing.isna().any() or frame.session_swing.duplicated().any():
        raise ValueError(f'Missing/duplicate swing identifier in {name}.')
if metadata[['user', 'session', 'hitter_side']].isna().any().any():
    raise ValueError('Missing athlete/session/side identity.')
meta = metadata.set_index('session_swing')
INPUT_HASHES = {p.name: file_hash(p) for p in [RAW / f'{n}.csv' for n in ARCHIVE_HASHES]
                + [RAW / 'metadata.csv', RAW / 'poi_metrics.csv']}
print('Trials:', len(metadata), 'metadata;', len(angles), 'angles;', len(landmarks), 'landmarks;', len(forces), 'force.')
print('Force: newtons; position: metres; angle: degrees. X toward pitcher; Z upward.')
print('Signed lead braking uses max(-lead_force_x, 0), not a horizontal resultant.')
print('Source model provenance: baseball_hitting/code/v3d/CMO.v3s computes MODEL_COG.')


In [ ]:
def scalar_event(frame, name):
    a = frame[name].to_numpy(float)
    if len(a) == 0 or not np.isfinite(a).all() or np.ptp(a) > 1e-7:
        raise ValueError('missing_or_inconsistent_' + name)
    return float(a[0])

def time_grid(frame, expected_hz):
    t = frame.time.to_numpy(float)
    if len(t) < 20 or not np.isfinite(t).all() or np.any(np.diff(t) <= 0):
        raise ValueError('invalid_time_grid')
    dt = np.median(np.diff(t))
    if not np.isclose(dt, 1 / expected_hz, atol=.00012, rtol=.04):
        raise ValueError('unexpected_sampling_rate')
    if not np.allclose(np.diff(t), dt, atol=.00012, rtol=.04):
        raise ValueError('irregular_grid_or_gap')
    return t

def interp_columns(t, values, q):
    v = np.asarray(values)
    if v.ndim == 1:
        return np.interp(q, t, v)
    return np.column_stack([np.interp(q, t, v[:, i]) for i in range(v.shape[1])])

def exact_window(t, values, start, stop):
    if not t[0] - 1e-8 <= start < stop <= t[-1] + 1e-8:
        raise ValueError('incomplete_window')
    if stop - start < 7 * np.median(np.diff(t)):
        raise ValueError('short_window')
    q = np.r_[start, t[(t > start + 1e-8) & (t < stop - 1e-8)], stop]
    return q, interp_columns(t, values, q)

def integral(y, t):
    return float(np.sum((y[:-1] + y[1:]) * np.diff(t) / 2))

def positive_integral(y, t):
    total = 0.
    for a, b, dt in zip(y[:-1], y[1:], np.diff(t)):
        if a >= 0 and b >= 0:
            total += (a + b) * dt / 2
        elif a > 0 > b:
            total += .5 * a * dt * a / (a - b)
        elif b > 0 > a:
            total += .5 * b * dt * b / (b - a)
    return float(total)

def crossing_count(y):
    return int(np.sum((y[:-1] < 0) & (y[1:] >= 0)))

def prepare_motion(a, l, side, contact, cutoff):
    if side not in ['R', 'L']:
        raise ValueError('invalid_hitter_side')
    ta, tl = time_grid(a, 360), time_grid(l, 360)
    if abs(scalar_event(a, 'contact_time') - scalar_event(l, 'contact_time')) > 1e-7:
        raise ValueError('motion_contact_disagreement')
    stop = contact - END_BEFORE_CONTACT_S
    begin = max(ta[0], tl[0], contact - 1.)
    if begin > contact - .60 or min(ta[-1], tl[-1]) < stop:
        raise ValueError('insufficient_motion_context')
    t = np.linspace(begin, stop, int(round((stop-begin)*360)) + 1)
    av = np.rad2deg(np.unwrap(np.deg2rad(a[JOINTS].to_numpy(float)), axis=0))
    lv = l[LM_COLS].to_numpy(float)
    ia = (ta >= begin - .01) & (ta <= stop + .01)
    il = (tl >= begin - .01) & (tl <= stop + .01)
    if not np.isfinite(av[ia]).all() or not np.isfinite(lv[il]).all():
        raise ValueError('nonfinite_motion_in_context')
    av, lv = interp_columns(ta, av, t), interp_columns(tl, lv, t)
    if not np.isfinite(av).all() or not np.isfinite(lv).all():
        raise ValueError('nonfinite_motion_after_interpolation')
    sos = butter(4, cutoff, fs=1/np.median(np.diff(t)), output='sos')
    av = sosfiltfilt(sos, av, axis=0)
    lv = sosfiltfilt(sos, lv, axis=0)
    coords = {b: lv[:, i*3:(i+1)*3].copy() for i, b in enumerate(LANDMARK_BASES)}
    if not .3 < np.median((coords['left_hip'][:, 2] + coords['right_hip'][:, 2])/2) < 1.6:
        raise ValueError('landmark_unit_or_pelvis_height_failure')
    if not .3 < np.median(coords['centerofmass'][:, 2]) < 1.8:
        raise ValueError('com_unit_or_height_failure')
    if side == 'L':
        for xyz in coords.values():
            xyz[:, 1] *= -1
    coords['pelvis'] = (coords['left_hip'] + coords['right_hip']) / 2
    coords['lead_ankle'] = coords['lajc'] if side == 'R' else coords['rajc']
    coords['rear_ankle'] = coords['rajc'] if side == 'R' else coords['lajc']
    av_rate = np.gradient(av, t, axis=0, edge_order=2)
    return {'time': t, 'angles': av, 'rates': av_rate, **coords}

def kinematic_landing(m, contact, height_tolerance=.010):
    t, ankle = m['time'], m['lead_ankle']
    vz = np.gradient(ankle[:, 2], t, edge_order=2)
    search = (t >= contact - .45) & (t <= contact - .03)
    ids = np.flatnonzero(search)
    if len(ids) < 12:
        return np.nan, 'insufficient_detector_window'
    apex = ids[np.argmax(ankle[ids, 2])]
    floor = np.quantile(ankle[ids, 2], .10)
    if ankle[apex, 2] - floor < .015:
        return np.nan, 'no_clear_ankle_lift'
    candidates = ids[(ids > apex) & (ankle[ids, 2] <= floor + height_tolerance) & (np.abs(vz[ids]) <= .25)]
    for idx in candidates:
        future = (t >= t[idx]) & (t <= t[idx] + .020)
        if future.sum() >= 6 and np.all(ankle[future, 2] <= floor + height_tolerance):
            return float(t[idx]), 'detected'
    return np.nan, 'no_sustained_settling'

def motion_features(m, start, stop, height):
    t = m['time']
    q, av = exact_window(t, m['angles'], start, stop)
    rate = interp_columns(t, m['rates'], q)
    row = {}
    for j, name in enumerate(JOINTS[:4]):
        label = name.replace('_angle_x', '')
        row[label + '_start_deg'] = av[0, j]
        row[label + '_change_deg'] = av[-1, j] - av[0, j]
        row[label + '_peak_extension_dps'] = np.max(-rate[:, j])
        row[label + '_extension_peak_ms'] = 1000 * (q[np.argmax(-rate[:, j])] - stop)
    for name in ['pelvis', 'centerofmass']:
        xyz = m[name]
        v = np.gradient(xyz, t, axis=0, edge_order=2)
        acc = np.gradient(v, t, axis=0, edge_order=2)
        p = interp_columns(t, xyz, q)
        vel = interp_columns(t, v, q)
        ac = interp_columns(t, acc, q)
        prefix = 'com' if name == 'centerofmass' else name
        for j, axis in enumerate('xz'):
            col = 0 if axis == 'x' else 2
            row[f'{prefix}_{axis}_change_pct_height'] = 100 * (p[-1, col] - p[0, col]) / height
            row[f'{prefix}_{axis}_peak_velocity_mps'] = np.max(vel[:, col])
            row[f'{prefix}_{axis}_min_accel_mps2'] = np.min(ac[:, col])
            row[f'{prefix}_{axis}_max_accel_mps2'] = np.max(ac[:, col])
    lead = interp_columns(t, m['lead_ankle'], q)
    rear = interp_columns(t, m['rear_ankle'], q)
    pelvis = interp_columns(t, m['pelvis'], q)
    row['stance_width_pct_height'] = 100 * np.linalg.norm(lead[0, :2] - rear[0, :2]) / height
    row['pelvis_to_lead_x_pct_height'] = 100 * (pelvis[0, 0] - lead[0, 0]) / height
    row['pelvis_rotation_peak_dps'] = np.max(rate[:, 4])
    row['pelvis_rotation_peak_ms'] = 1000 * (q[np.argmax(rate[:, 4])] - stop)
    row['torso_rotation_peak_dps'] = np.max(rate[:, 5])
    row['rotation_peak_gap_ms'] = 1000 * (q[np.argmax(rate[:, 5])] - q[np.argmax(rate[:, 4])])
    row['duration_ms'] = 1000 * (stop - start)
    if not np.isfinite(list(row.values())).all():
        raise ValueError('nonfinite_features')
    return row

def force_targets(f, start, stop, mass, contact):
    t = time_grid(f, 1080)
    if abs(scalar_event(f, 'contact_time') - contact) > 1e-7:
        raise ValueError('force_motion_contact_disagreement')
    q, v = exact_window(t, f[FORCE_COLS].to_numpy(float), start, stop)
    if not np.isfinite(v).all():
        raise ValueError('nonfinite_force_window')
    v = v / (mass * G)
    lead_x, lead_z, rear_z = v[:, 0], v[:, 2], v[:, 5]
    total_z = lead_z + rear_z
    if min(lead_z.min(), rear_z.min()) < -.10 or total_z.max() > 5.:
        raise ValueError('implausible_vertical_force_window')
    peak = int(np.argmax(lead_z))
    row = {'lead_peak_vertical_bw': float(lead_z[peak]),
           'lead_braking_impulse_bws': positive_integral(-lead_x, q),
           'lead_peak_time_ms': float(1000 * (q[peak] - contact)),
           'total_peak_vertical_bw': float(total_z.max()),
           'force_peak_at_boundary': bool(peak in [0, len(q)-1]),
           'lead_min_vertical_bw': float(lead_z.min()),
           'total_min_vertical_bw': float(total_z.min())}
    return row, (q, lead_z, total_z, lead_x)

def numerical_checks():
    t = np.linspace(0, 1, 1081)
    q, v = exact_window(t, 2*t + 1, .1234, .8765)
    assert np.allclose(v[[0, -1]], 2*np.array([.1234, .8765]) + 1)
    assert np.isclose(integral(v, q), (q[-1]**2+q[-1]) - (q[0]**2+q[0]))
    assert np.isclose(positive_integral(np.array([-1., 1.]), np.array([0., 1.])), .25)
    assert np.isclose(positive_integral(np.array([1., -1.]), np.array([0., 1.])), .25)
    assert np.isclose(positive_integral(np.array([-1., -1.]), np.array([0., 1.])), 0.)
    a = np.rad2deg(np.unwrap(np.deg2rad([170., 179., -179., -170.])))
    assert np.isclose(a[-1] - a[0], 20.)
    try:
        exact_window(t, t, -.1, .5)
        raise AssertionError('Out-of-range window was accepted.')
    except ValueError:
        pass
    mass, contact = 70., .8
    ft = np.arange(1081)/1080
    f = pd.DataFrame({'time': ft, 'contact_time': contact})
    for name in FORCE_COLS:
        f[name] = 0.
    f['lead_force_z'] = 1.2*mass*G
    f['rear_force_z'] = .4*mass*G
    f['lead_force_x'] = -.3*mass*G
    targets, _ = force_targets(f, .65, .79, mass, contact)
    assert np.isclose(targets['lead_peak_vertical_bw'], 1.2)
    assert np.isclose(targets['total_peak_vertical_bw'], 1.6)
    assert np.isclose(targets['lead_braking_impulse_bws'], .3*.14)
    assert np.isclose(targets['lead_peak_time_ms'], -150.)
    mt = np.arange(361)/360
    a = pd.DataFrame({'time': mt, 'contact_time': contact, 'fp_10_time': .5, 'fp_100_time': .6})
    l = a.copy()
    for name in JOINTS:
        a[name] = mt * 10
    for name in LANDMARK_BASES:
        for axis in 'xyz':
            l[f'{name}_{axis}'] = (1. if axis == 'z' else 0.) + mt*.01
    one = prepare_motion(a, l, 'R', contact, 12.)
    a[['fp_10_time', 'fp_100_time']] = np.nan
    l[['fp_10_time', 'fp_100_time']] = np.nan
    two = prepare_motion(a, l, 'R', contact, 12.)
    assert np.array_equal(one['angles'], two['angles'])
    assert np.allclose(np.diff(one['time']), np.diff(one['time'])[0])
    mirrored = prepare_motion(a, l, 'L', contact, 12.)
    assert np.allclose(mirrored['centerofmass'][:, 1], -one['centerofmass'][:, 1])
    assert np.allclose(mirrored['centerofmass'][:, 0], one['centerofmass'][:, 0])
    print('Numerical checks passed: integration, normalization, force signs, timing, uniform derivatives, mirroring, and no primary foot-event dependency.')

numerical_checks()


In [ ]:
CONFIGS = [('primary', .150, 12.), ('filter8', .150, 8.), ('filter16', .150, 16.),
           ('window120', .120, 12.), ('window180', .180, 12.)]
records, eligibility, event_records, physics_records = [], [], [], []
motion_cache, trace_cache = {}, {}
TARGETS = ['lead_peak_vertical_bw', 'lead_braking_impulse_bws', 'lead_peak_time_ms', 'total_peak_vertical_bw']
for swing, r in meta.iterrows():
    ident = {'session_swing': swing, 'hitter_id': r.user, 'session_id': r.session, 'hitter_side': r.hitter_side}
    missing = [name for name, source in [('angles', angles), ('landmarks', landmarks), ('force', forces)] if swing not in source]
    if missing:
        eligibility.append({**ident, 'config': 'all', 'status': 'missing_' + '_'.join(missing)})
        continue
    mass, height = float(r.session_mass_lbs)*.45359237, float(r.session_height_in)*.0254
    if not np.isfinite([mass, height]).all() or not 30 < mass < 200 or not 1.2 < height < 2.3:
        eligibility.append({**ident, 'config': 'all', 'status': 'invalid_anthropometry'})
        continue
    a, l, f = angles[swing], landmarks[swing], forces[swing]
    try:
        contact = scalar_event(a, 'contact_time')
    except ValueError as exc:
        eligibility.append({**ident, 'config': 'all', 'status': str(exc)})
        continue
    motions = {}
    for config, width, cutoff in CONFIGS:
        try:
            if cutoff not in motions:
                motions[cutoff] = prepare_motion(a, l, r.hitter_side, contact, cutoff)
            m = motions[cutoff]
            start, stop = contact-width, contact-END_BEFORE_CONTACT_S
            features = motion_features(m, start, stop, height)
            targets, force_trace = force_targets(f, start, stop, mass, contact)
            row = {**ident, 'config': config, 'mass_kg': mass, 'height_m': height,
                   'left_handed': int(r.hitter_side == 'L'), **features, **targets}
            records.append(row)
            eligibility.append({**ident, 'config': config, 'status': 'retained'})
            if config == 'primary':
                motion_cache[swing] = m
                trace_cache[swing] = (contact, force_trace)
                detection, detection_status = kinematic_landing(m, contact)
                ev = {**ident, 'kinematic_landing_time': detection, 'detector_status': detection_status,
                      'contact_time': contact}
                for event in EVENTS[:2]:
                    try:
                        ev[event] = scalar_event(a, event)
                    except ValueError:
                        ev[event] = np.nan
                for tol in [.005, .015]:
                    ev[f'kinematic_landing_{int(tol*1000)}mm'] = kinematic_landing(m, contact, tol)[0]
                event_records.append(ev)
                q, lead_z, total_z, lead_x = force_trace
                com_v = np.gradient(m['centerofmass'][:, 2], m['time'], edge_order=2)
                com_a = np.gradient(com_v, m['time'], edge_order=2)
                approx = 1 + interp_columns(m['time'], com_a, q) / G
                physics_records.append({**ident, 'approx_total_peak_bw': float(approx.max()),
                                        'measured_total_peak_bw': float(total_z.max()),
                                        'curve_mae_bw': integral(np.abs(approx-total_z), q)/(q[-1]-q[0]),
                                        'curve_bias_bw': integral(approx-total_z, q)/(q[-1]-q[0])})
        except ValueError as exc:
            eligibility.append({**ident, 'config': config, 'status': str(exc)})
measurements = pd.DataFrame(records)
audit = pd.DataFrame(eligibility)
events = pd.DataFrame(event_records)
physics = pd.DataFrame(physics_records)
if measurements.empty:
    raise ValueError('No eligible swings; inspect source schemas, event coverage, and units.')
primary = measurements[measurements.config.eq('primary')].copy().reset_index(drop=True)
outcome_cols = ['bat_speed_mph_contact_x', 'blast_bat_speed_mph_x', 'exit_velo_mph_x']
primary = primary.merge(poi[['session_swing', *outcome_cols]], on='session_swing', how='left', validate='one_to_one')
for frame, filename in [(measurements, 'swing_measurements.csv'), (audit, 'eligibility_audit.csv'),
                         (events, 'landing_event_audit.csv'), (physics, 'com_physics_check.csv')]:
    frame.to_csv(OUT / filename, index=False)
display(audit.groupby(['config', 'status']).size().rename('swings').reset_index())
print('Primary:', len(primary), 'swings from', primary.hitter_id.nunique(), 'hitters.')
print('Primary targets describe only the specified window, not whole-swing force maxima or whole-swing impulse.')
print('Knee/hip extension rates are negative derivatives of flexion angles; pelvis rotation is a Cardan-angle rate proxy.')
print('Published trajectories are already processed. Additional filters and finite differences change the operational measurement.')


In [ ]:
CONTROLS = ['height_m', 'mass_kg', 'left_handed']
MOTION_FEATURES = [c for c in primary.columns if c.startswith(('lead_knee_', 'rear_knee_', 'lead_hip_', 'rear_hip_', 'pelvis_x_', 'pelvis_z_'))]
MOTION_FEATURES += ['stance_width_pct_height', 'pelvis_to_lead_x_pct_height']
COM_FEATURES = [c for c in primary.columns if c.startswith('com_')]
MODEL_FEATURES = {'mean': [], 'controls': CONTROLS,
                  'lower_body': CONTROLS + MOTION_FEATURES,
                  'lower_body_com': CONTROLS + MOTION_FEATURES + COM_FEATURES}
for cols in MODEL_FEATURES.values():
    assert not set(cols) & set(TARGETS + outcome_cols + ['fp_10_time', 'fp_100_time', 'duration_ms'])
assert not any('rotation' in c for c in MOTION_FEATURES)

def equal_hitter_weights(groups):
    groups = np.asarray(groups)
    _, inv, counts = np.unique(groups, return_inverse=True, return_counts=True)
    weights = 1. / counts[inv]
    return weights / weights.mean()

def grouped_splits(groups, n_splits, seed):
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) < n_splits:
        raise ValueError(f'Need at least {n_splits} hitters; got {len(unique)}.')
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(unique)
    for fold, test_groups in enumerate(np.array_split(shuffled, n_splits)):
        test = np.flatnonzero(np.isin(groups, test_groups))
        train = np.flatnonzero(~np.isin(groups, test_groups))
        assert not set(groups[train]) & set(groups[test])
        yield fold, train, test

def hitter_error(actual, predicted, groups, squared=False):
    err = (np.asarray(predicted)-np.asarray(actual))**2 if squared else np.abs(np.asarray(predicted)-np.asarray(actual))
    return float(pd.DataFrame({'id': groups, 'e': err}).groupby('id').e.mean().mean())

def fit_predict(x_train, y_train, groups_train, x_test, alpha):
    weights = equal_hitter_weights(groups_train)
    scaler = StandardScaler().fit(x_train, sample_weight=weights)
    model = Ridge(alpha=alpha).fit(scaler.transform(x_train), y_train, sample_weight=weights)
    return model.predict(scaler.transform(x_test))

def nested_predictions(table, targets, feature_sets, repeats=CV_REPEATS, run_name='primary', outer_assignment=None):
    prediction_rows, tuning_rows, fold_rows = [], [], []
    for target in targets:
        required = list(dict.fromkeys([target, *[c for cols in feature_sets.values() for c in cols]]))
        data = table.replace([np.inf, -np.inf], np.nan).dropna(subset=required).reset_index(drop=True)
        groups = data.hitter_id.to_numpy()
        y = data[target].to_numpy(float)
        if data.hitter_id.nunique() < max(OUTER_FOLDS+1, INNER_FOLDS+2):
            print('Skipping insufficient hitter cohort:', run_name, target)
            continue
        for repeat in range(repeats):
            if outer_assignment is None:
                splits = grouped_splits(groups, OUTER_FOLDS, SEED + repeat)
            else:
                ref = outer_assignment[outer_assignment.target.eq(target) & outer_assignment.repeat.eq(repeat)]
                if ref.session_swing.duplicated().any():
                    raise ValueError('Duplicate reference fold assignment.')
                assignment = data.session_swing.map(ref.set_index('session_swing').fold)
                if assignment.isna().any():
                    raise ValueError('Screened trial absent from original outer-fold assignment.')
                check = pd.DataFrame({'hitter_id': groups, 'fold': assignment}).groupby('hitter_id').fold.nunique()
                assert check.eq(1).all()
                splits = [(int(fold), np.flatnonzero(assignment.to_numpy()!=fold),
                           np.flatnonzero(assignment.to_numpy()==fold)) for fold in sorted(ref.fold.unique())]
            for fold, tr, te in splits:
                assert not set(groups[tr]) & set(groups[te])
                if len(tr) == 0 or len(te) == 0:
                    raise ValueError('Empty fold after sensitivity screening.')
                for idx in te:
                    fold_rows.append({'run': run_name, 'target': target, 'repeat': repeat, 'fold': fold,
                                      'session_swing': data.session_swing.iloc[idx], 'hitter_id': groups[idx]})
                for model_name, cols in feature_sets.items():
                    if not cols:
                        pred = np.repeat(np.average(y[tr], weights=equal_hitter_weights(groups[tr])), len(te))
                        best_alpha = np.nan
                    else:
                        x = data[cols].to_numpy(float)
                        scores = []
                        for alpha in ALPHAS:
                            inner_pred = np.full(len(tr), np.nan)
                            for _, itr, ite in grouped_splits(groups[tr], INNER_FOLDS, SEED + 100 + repeat*10 + fold):
                                inner_pred[ite] = fit_predict(x[tr][itr], y[tr][itr], groups[tr][itr], x[tr][ite], alpha)
                            score = hitter_error(y[tr], inner_pred, groups[tr])
                            scores.append(score)
                            tuning_rows.append({'run': run_name, 'target': target, 'model': model_name,
                                                'repeat': repeat, 'fold': fold, 'alpha': alpha, 'inner_mae': score})
                        best_alpha = ALPHAS[int(np.argmin(scores))]
                        pred = fit_predict(x[tr], y[tr], groups[tr], x[te], best_alpha)
                    for idx, estimate in zip(te, pred):
                        prediction_rows.append({'run': run_name, 'target': target, 'model': model_name,
                                                'repeat': repeat, 'fold': fold, 'alpha': best_alpha,
                                                'session_swing': data.session_swing.iloc[idx], 'hitter_id': groups[idx],
                                                'actual': y[idx], 'predicted': estimate})
        print('Validated', run_name, target, ':', len(data), 'swings,', data.hitter_id.nunique(), 'hitters.', flush=True)
    return pd.DataFrame(prediction_rows), pd.DataFrame(tuning_rows), pd.DataFrame(fold_rows)

def model_metrics(predictions):
    rows = []
    for (run, target, model), d in predictions.groupby(['run', 'target', 'model']):
        d = d.copy()
        d['abs_error'] = (d.predicted-d.actual).abs()
        d['sq_error'] = (d.predicted-d.actual)**2
        d['bias'] = d.predicted-d.actual
        means = d.groupby('hitter_id')[['abs_error', 'sq_error', 'bias']].mean()
        repeats = [g.groupby('hitter_id').abs_error.mean().mean() for _, g in d.groupby('repeat')]
        rows.append({'run': run, 'target': target, 'model': model, 'swings': d.session_swing.nunique(),
                     'hitters': len(means), 'mae': means.abs_error.mean(), 'rmse': np.sqrt(means.sq_error.mean()),
                     'bias': means.bias.mean(), 'repeat_mae_min': min(repeats), 'repeat_mae_max': max(repeats)})
    return pd.DataFrame(rows)

def paired_improvements(predictions, candidate='lower_body_com', reference='controls'):
    rows = []
    for (run, target), d in predictions.groupby(['run', 'target']):
        d = d.assign(error=(d.predicted-d.actual).abs())
        e = d.groupby(['hitter_id', 'model']).error.mean().unstack('model')
        if not {candidate, reference}.issubset(e.columns):
            continue
        delta = (e[reference]-e[candidate]).dropna().to_numpy(float)
        rng = np.random.default_rng(SEED)
        samples = rng.choice(delta, size=(BOOTSTRAPS, len(delta)), replace=True).mean(axis=1)
        rows.append({'run': run, 'target': target, 'candidate': candidate, 'reference': reference,
                     'hitters': len(delta), 'mae_improvement': delta.mean(),
                     'low': np.quantile(samples, .025), 'high': np.quantile(samples, .975)})
    return pd.DataFrame(rows)

def validation_checks():
    ids = np.array(['a']*3 + ['b'] + ['c']*2 + ['d']*4 + ['e']*2 + ['f']*2 + ['g']*2)
    weights = equal_hitter_weights(ids)
    totals = pd.DataFrame({'id': ids, 'w': weights}).groupby('id').w.sum()
    assert np.allclose(totals, totals.iloc[0])
    for _, tr, te in grouped_splits(ids, 3, SEED):
        assert not set(ids[tr]) & set(ids[te])
        for _, itr, ite in grouped_splits(ids[tr], 2, SEED):
            assert not set(ids[tr][itr]) & set(ids[tr][ite])
    s = StandardScaler().fit(np.array([[0.], [2.]]))
    assert np.isclose(s.mean_[0], 1.)
    s.transform(np.array([[100000.]]))
    assert np.isclose(s.mean_[0], 1.)
    print('Validation checks passed: hitter-disjoint outer/inner folds, equal hitter weights, training-only scaler.')

validation_checks()
predictions, tuning, folds = nested_predictions(primary, TARGETS, MODEL_FEATURES)
metrics = model_metrics(predictions)
improvements = pd.concat([paired_improvements(predictions, 'lower_body', 'controls'),
                          paired_improvements(predictions),
                          paired_improvements(predictions, 'lower_body_com', 'lower_body')], ignore_index=True)
for frame, filename in [(predictions, 'held_out_predictions.csv'), (tuning, 'inner_tuning.csv'),
                         (folds, 'outer_fold_membership.csv'), (metrics, 'model_metrics.csv'),
                         (improvements, 'paired_hitter_bootstrap.csv')]:
    frame.to_csv(OUT / filename, index=False)
display(metrics)
display(improvements)
print('Positive improvement means lower held-out MAE. Intervals resample fixed per-hitter errors; training is not rerun.')
print('No truncation of negative model estimates. Bias and unphysical predictions remain visible.')


In [ ]:
event_summary = []
for event in ['fp_10_time', 'fp_100_time']:
    delta = 1000 * (events.kinematic_landing_time - events[event])
    valid = delta.dropna()
    event_summary.append({'comparison': 'kinematic_minus_' + event, 'n': len(valid),
                          'median_ms': valid.median(), 'median_absolute_ms': valid.abs().median(),
                          'p10_ms': valid.quantile(.10), 'p90_ms': valid.quantile(.90)})
events['official_chronology_valid'] = (events.fp_10_time < events.fp_100_time) & (events.fp_100_time < events.contact_time)
event_assisted_rows = []
kinematic_rows = []
for _, e in events.iterrows():
    swing = e.session_swing
    r = primary.loc[primary.session_swing.eq(swing)].iloc[0]
    stop = e.contact_time - END_BEFORE_CONTACT_S
    if np.isfinite(e.kinematic_landing_time) and .030 <= stop - e.kinematic_landing_time <= .300:
        try:
            feat = motion_features(motion_cache[swing], e.kinematic_landing_time, stop, r.height_m)
            kinematic_rows.append({**r.to_dict(), **{'kinematic_' + k: v for k, v in feat.items()}})
        except ValueError:
            pass
    if not e.official_chronology_valid:
        continue
    if not .030 <= stop - e.fp_100_time <= .300:
        continue
    try:
        feat = motion_features(motion_cache[swing], e.fp_100_time, stop, r.height_m)
        event_assisted_rows.append({**r.to_dict(), **{'oracle_' + k: v for k, v in feat.items()}})
    except ValueError:
        continue
oracle = pd.DataFrame(event_assisted_rows)
kinematic = pd.DataFrame(kinematic_rows)
if len(kinematic) and kinematic.hitter_id.nunique() >= 12:
    kinematic_cols = ['kinematic_' + c for c in MOTION_FEATURES + COM_FEATURES] + ['kinematic_duration_ms']
    kinematic_sets = {'controls': CONTROLS, 'lower_body_com': MODEL_FEATURES['lower_body_com'],
                      'kinematic_landing': CONTROLS + kinematic_cols}
    kp, kt, kf = nested_predictions(kinematic, [TARGETS[0]], kinematic_sets, repeats=1, run_name='kinematic_landing_common_cohort')
    kp.to_csv(OUT / 'kinematic_landing_predictions.csv', index=False)
    kt.to_csv(OUT / 'kinematic_landing_tuning.csv', index=False)
    kf.to_csv(OUT / 'kinematic_landing_folds.csv', index=False)
    model_metrics(kp).to_csv(OUT / 'kinematic_landing_metrics.csv', index=False)
    display(model_metrics(kp))
oracle_pred = pd.DataFrame()
if len(oracle) and oracle.hitter_id.nunique() >= 12:
    oracle_cols = ['oracle_' + c for c in MOTION_FEATURES + COM_FEATURES] + ['oracle_duration_ms']
    oracle_sets = {'controls': CONTROLS, 'lower_body_com': MODEL_FEATURES['lower_body_com'],
                   'force_assisted_plant': CONTROLS + oracle_cols}
    oracle_pred, oracle_tuning, oracle_folds = nested_predictions(
        oracle, [TARGETS[0]], oracle_sets, repeats=1, run_name='force_assisted_common_cohort')
    oracle_pred.to_csv(OUT / 'force_assisted_predictions.csv', index=False)
    oracle_tuning.to_csv(OUT / 'force_assisted_tuning.csv', index=False)
    oracle_folds.to_csv(OUT / 'force_assisted_folds.csv', index=False)
    display(model_metrics(oracle_pred))
    model_metrics(oracle_pred).to_csv(OUT / 'force_assisted_metrics.csv', index=False)
pd.DataFrame(event_summary).to_csv(OUT / 'landing_detector_agreement.csv', index=False)
display(pd.DataFrame(event_summary))
print(events.detector_status.value_counts().to_string())
print('Force-assisted comparison retains fixed-window targets. Only its feature window depends on official plant.')
print('It is not evidence of performance without force plates. A missing detector never excludes a primary trial.')
print('Kinematic and force-assisted cohorts differ. Compare models only within each common-cohort analysis.')


In [ ]:
sensitivity_rows, sensitivity_prediction_frames = [], []
for config, _, _ in CONFIGS[1:]:
    alt = measurements[measurements.config.eq(config)]
    matched = primary[['session_swing', *MOTION_FEATURES, *COM_FEATURES]].merge(
        alt[['session_swing', *MOTION_FEATURES, *COM_FEATURES]], on='session_swing', suffixes=('_primary', '_alt'), validate='one_to_one')
    for feature in MOTION_FEATURES + COM_FEATURES:
        x, y = matched[feature+'_primary'], matched[feature+'_alt']
        rho = spearmanr(x, y).statistic if x.nunique() > 1 and y.nunique() > 1 else np.nan
        sensitivity_rows.append({'config': config, 'feature': feature, 'common_swings': len(matched),
                                 'rank_correlation': rho, 'median_absolute_change': np.median(np.abs(x-y))})
    if RUN_MODEL_SENSITIVITY:
        ids = sorted(set(primary.session_swing) & set(alt.session_swing))
        for label, cohort in [('primary', primary), (config, alt)]:
            cohort = cohort[cohort.session_swing.isin(ids)].sort_values('session_swing').reset_index(drop=True)
            p, tu, fo = nested_predictions(cohort, [TARGETS[0]],
                {k: MODEL_FEATURES[k] for k in ['controls', 'lower_body_com']}, repeats=1,
                run_name=config + '_paired_' + label)
            sensitivity_prediction_frames.append(p)
            tu.to_csv(OUT / (config + '_paired_' + label + '_tuning.csv'), index=False)
            fo.to_csv(OUT / (config + '_paired_' + label + '_folds.csv'), index=False)
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUT / 'feature_sensitivity.csv', index=False)
sensitivity_predictions = pd.concat(sensitivity_prediction_frames, ignore_index=True) if sensitivity_prediction_frames else pd.DataFrame()
if len(sensitivity_predictions):
    sensitivity_predictions.to_csv(OUT / 'sensitivity_predictions.csv', index=False)
    model_metrics(sensitivity_predictions).to_csv(OUT / 'sensitivity_model_metrics.csv', index=False)
    display(model_metrics(sensitivity_predictions))
boundary = primary.groupby('hitter_id').force_peak_at_boundary.mean()
print('Hitter-weighted proportion of lead-force peaks at a window boundary:', round(float(boundary.mean()), 3))
print('Changing the window changes the target definition. Paired cohorts/folds support sensitivity review, not identical-target replication.')


In [ ]:
CLUSTER_FEATURES = ['lead_knee_change_deg', 'lead_knee_peak_extension_dps',
                    'rear_knee_change_deg', 'rear_hip_change_deg',
                    'pelvis_x_change_pct_height', 'pelvis_x_min_accel_mps2',
                    'pelvis_z_change_pct_height', 'stance_width_pct_height', 'pelvis_to_lead_x_pct_height']
PROFILE_OUTCOMES = [*TARGETS, 'pelvis_rotation_peak_dps', 'torso_rotation_peak_dps', *outcome_cols]
session_keys = ['hitter_id', 'session_id', 'hitter_side']
session_profiles = primary.groupby(session_keys)[CLUSTER_FEATURES + PROFILE_OUTCOMES].median()
session_profiles['n_swings'] = primary.groupby(session_keys).size()
session_profiles = session_profiles[session_profiles.n_swings >= MIN_PROFILE_SWINGS].reset_index()
profiles = session_profiles.groupby('hitter_id')[CLUSTER_FEATURES + PROFILE_OUTCOMES].mean()
profiles['n_sessions'] = session_profiles.groupby('hitter_id').size()
profiles['n_swings'] = session_profiles.groupby('hitter_id').n_swings.sum()
profiles = profiles.dropna(subset=CLUSTER_FEATURES)
if len(profiles) < 12:
    raise ValueError('Too few eligible hitter profiles for descriptive strategy analysis.')
scaler = StandardScaler().fit(profiles[CLUSTER_FEATURES])
x = scaler.transform(profiles[CLUSTER_FEATURES])
k_rows, cluster_solutions = [], {}
for k in range(2, min(5, len(profiles)//5) + 1):
    solution = KMeans(n_clusters=k, n_init=30, random_state=SEED).fit(x)
    sizes = np.bincount(solution.labels_, minlength=k)
    score = silhouette_score(x, solution.labels_)
    k_rows.append({'k': k, 'silhouette': score, 'smallest_cluster': int(sizes.min())})
    cluster_solutions[k] = solution
k_table = pd.DataFrame(k_rows)
eligible = k_table[k_table.smallest_cluster >= 5]
if eligible.empty:
    chosen_k = int(k_table.loc[k_table.silhouette.idxmax(), 'k'])
else:
    chosen_k = int(eligible.loc[eligible.silhouette.idxmax(), 'k'])
solution = cluster_solutions[chosen_k]
profiles['cluster'] = solution.labels_ + 1
stability_rows, assignment_rows = [], []
rng = np.random.default_rng(SEED)
grouped_primary = {h: g for h, g in primary.groupby('hitter_id')}
for b in range(CLUSTER_BOOTSTRAPS):
    resampled = []
    for hitter in profiles.index:
        g = grouped_primary[hitter]
        per_session = []
        for _, session in g.groupby(['session_id', 'hitter_side']):
            if len(session) < MIN_PROFILE_SWINGS:
                continue
            take = rng.integers(0, len(session), len(session))
            per_session.append(session.iloc[take][CLUSTER_FEATURES].median().to_numpy(float))
        resampled.append(np.mean(per_session, axis=0))
    boot_x = StandardScaler().fit_transform(np.asarray(resampled))
    boot_labels = KMeans(n_clusters=chosen_k, n_init=10, random_state=SEED+b+1).fit_predict(boot_x)
    ari = adjusted_rand_score(solution.labels_, boot_labels)
    stability_rows.append({'resample': b, 'adjusted_rand_index': ari})
    for i, hitter in enumerate(profiles.index):
        original_peers = solution.labels_ == solution.labels_[i]
        bootstrap_peers = boot_labels == boot_labels[i]
        union = np.sum(original_peers | bootstrap_peers) - 1
        intersection = np.sum(original_peers & bootstrap_peers) - 1
        assignment_rows.append({'resample': b, 'hitter_id': hitter,
                                'peer_jaccard': intersection/union if union > 0 else 1.})
stability = pd.DataFrame(stability_rows)
assignment_stability = pd.DataFrame(assignment_rows).groupby('hitter_id').peer_jaccard.mean()
profiles['peer_stability'] = assignment_stability
median_ari = float(stability.adjusted_rand_index.median())
silhouette = float(k_table.loc[k_table.k.eq(chosen_k), 'silhouette'].iloc[0])
CLUSTER_STATUS = 'descriptive separation with reasonable resampling stability' if median_ari >= .60 and silhouette >= .25 else 'weak or unstable separation; emphasize continuous profiles'
cluster_centers = profiles.groupby('cluster')[CLUSTER_FEATURES].mean()
standardized_centers = pd.DataFrame(scaler.transform(cluster_centers), index=cluster_centers.index, columns=CLUSTER_FEATURES)
cluster_outcomes = profiles.groupby('cluster')[PROFILE_OUTCOMES].agg(['mean', 'median', 'count'])
for frame, filename in [(session_profiles, 'session_profiles.csv'), (profiles.reset_index(), 'hitter_profiles.csv'),
                         (k_table, 'cluster_k_review.csv'), (stability, 'cluster_resampling_stability.csv'),
                         (standardized_centers.reset_index(), 'cluster_standardized_centers.csv')]:
    frame.to_csv(OUT / filename, index=False)
cluster_outcomes.to_csv(OUT / 'cluster_outcomes_descriptive.csv')
display(k_table)
print('Chosen k:', chosen_k, '; median resampling ARI:', round(median_ari, 3), ';', CLUSTER_STATUS)
print('The stability checks resample swings within existing hitter-sessions. They do not establish across-day reliability.')
print('Cluster k is exploratory. No force, rotation-speed, bat-speed, or exit-velocity outcome enters clustering.')
print('Cluster comparisons are post-selection descriptions, not independent hypothesis tests or coaching prescriptions.')
display(cluster_outcomes)


In [ ]:
association_rows = []
rng = np.random.default_rng(SEED)
for feature in CLUSTER_FEATURES:
    for outcome in ['pelvis_rotation_peak_dps', 'torso_rotation_peak_dps', 'bat_speed_mph_contact_x', 'lead_peak_vertical_bw']:
        data = profiles[[feature, outcome]].dropna()
        if len(data) < 10 or min(data.nunique()) < 2:
            continue
        values = data.to_numpy(float)
        rho = float(spearmanr(values[:, 0], values[:, 1]).statistic)
        boot = []
        for _ in range(1000):
            v = values[rng.integers(0, len(values), len(values))]
            if np.ptp(v[:, 0]) > 0 and np.ptp(v[:, 1]) > 0:
                boot.append(spearmanr(v[:, 0], v[:, 1]).statistic)
        association_rows.append({'feature': feature, 'outcome': outcome, 'hitters': len(values),
                                 'spearman_rho': rho, 'low': np.quantile(boot, .025), 'high': np.quantile(boot, .975)})
associations = pd.DataFrame(association_rows)
associations.to_csv(OUT / 'exploratory_hitter_associations.csv', index=False)
print('Associations are unadjusted exploratory comparisons; the intervals are not multiple-testing adjusted.')
print('Rotation outcomes are angle derivatives, not measured rotational power or torque.')
display(associations[associations.outcome.eq('pelvis_rotation_peak_dps')])


In [ ]:
TARGET_LABELS = {'lead_peak_vertical_bw': 'Lead vertical peak (BW)',
                 'lead_braking_impulse_bws': 'Lead braking impulse (BW·s)',
                 'lead_peak_time_ms': 'Lead vertical peak timing (ms vs contact)',
                 'total_peak_vertical_bw': 'Total vertical peak (BW)'}
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, target in zip(axes.flat, TARGETS):
    d = predictions[(predictions.target.eq(target)) & predictions.model.eq('lower_body_com')]
    d = d.groupby(['session_swing', 'hitter_id'])[['actual', 'predicted']].mean().reset_index()
    ax.scatter(d.actual, d.predicted, alpha=.45, s=18, color='#167d9a')
    low, high = min(d.actual.min(), d.predicted.min()), max(d.actual.max(), d.predicted.max())
    ax.plot([low, high], [low, high], '--', color='#555555')
    ax.set(title=TARGET_LABELS[target], xlabel='Measured', ylabel='Held-out estimate')
fig.suptitle('Repeat-averaged held-out estimates — lower body + modeled COM', fontsize=15)
fig.tight_layout()
fig.savefig(FIG / '01_held_out_estimates.png', dpi=160, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, target in zip(axes.flat, TARGETS):
    g = metrics[metrics.target.eq(target)].set_index('model').reindex(list(MODEL_FEATURES))
    ax.bar(np.arange(len(g)), g.mae, color=['#b7bdc6', '#8293a6', '#167d9a', '#d17a39'])
    ax.set_xticks(np.arange(len(g)), ['Mean', 'Body controls', 'Lower body', '+ modeled COM'], rotation=15)
    ax.set(title=TARGET_LABELS[target], ylabel='Hitter-weighted MAE')
fig.suptitle('Nested validation with entire hitters held out', fontsize=15)
fig.tight_layout()
fig.savefig(FIG / '02_model_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(12, max(3.5, chosen_k*.75)))
im = ax.imshow(standardized_centers.to_numpy(), cmap='RdBu_r', vmin=-2, vmax=2, aspect='auto')
ax.set_xticks(range(len(CLUSTER_FEATURES)), [s.replace('_', ' ') for s in CLUSTER_FEATURES], rotation=55, ha='right')
ax.set_yticks(range(chosen_k), ['Strategy ' + str(i) for i in standardized_centers.index])
ax.set_title('Descriptive movement strategies — sample-relative standardized values')
fig.colorbar(im, ax=ax, label='Sample-relative SD')
fig.tight_layout()
fig.savefig(FIG / '03_movement_strategies.png', dpi=160, bbox_inches='tight')
plt.show()

pc = PCA(n_components=2).fit(x)
scores = pc.transform(x)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for label in range(chosen_k):
    take = solution.labels_ == label
    axes[0].scatter(scores[take, 0], scores[take, 1], label=f'Strategy {label+1}', s=35, alpha=.8)
axes[0].set(xlabel=f'PC1 ({pc.explained_variance_ratio_[0]:.0%})', ylabel=f'PC2 ({pc.explained_variance_ratio_[1]:.0%})', title='Continuous profiles with descriptive clusters')
axes[0].legend(fontsize=8)
axes[1].hist(stability.adjusted_rand_index, bins=15, color='#167d9a', edgecolor='white')
axes[1].axvline(.60, color='#d17a39', linestyle='--', label='Pragmatic review threshold')
axes[1].set(xlabel='Adjusted Rand index after swing resampling', ylabel='Resamples', title=CLUSTER_STATUS)
axes[1].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / '04_profile_separation_stability.png', dpi=160, bbox_inches='tight')
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(physics.measured_total_peak_bw, physics.approx_total_peak_bw, s=16, alpha=.45, color='#167d9a')
lo = min(physics.measured_total_peak_bw.min(), physics.approx_total_peak_bw.min())
hi = max(physics.measured_total_peak_bw.max(), physics.approx_total_peak_bw.max())
axes[0].plot([lo, hi], [lo, hi], '--', color='#555555')
axes[0].set(xlabel='Measured total peak (BW)', ylabel='1 + modeled COM vertical acceleration / g', title='Approximate physics check; not individual-foot force')
delta = 1000*(events.kinematic_landing_time-events.fp_10_time)
axes[1].hist(delta.dropna(), bins=25, color='#d17a39', edgecolor='white')
axes[1].axvline(0, color='#555555', linestyle='--')
axes[1].set(xlabel='Kinematic settling minus 10% BW event (ms)', ylabel='Swings', title='Independent landing detector agreement')
fig.tight_layout()
fig.savefig(FIG / '05_com_and_event_checks.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
def show_hitter(hitter_id=None):
    if hitter_id is None:
        hitter_id = profiles.sort_values(['n_swings', 'peer_stability'], ascending=False).index[0]
    hitter_id = str(hitter_id)
    if hitter_id not in profiles.index:
        raise KeyError('Choose a hitter_id from hitter_profiles.csv, preserving leading zeros.')
    profile = profiles.loc[hitter_id]
    d = primary[primary.hitter_id.eq(hitter_id)]
    print('Hitter', hitter_id, '| swings:', int(profile.n_swings), '| descriptive strategy:', int(profile.cluster),
          '| peer stability:', round(float(profile.peer_stability), 2))
    percentiles = profiles[CLUSTER_FEATURES].rank(pct=True).loc[hitter_id] * 100
    display(pd.DataFrame({'value': profile[CLUSTER_FEATURES], 'sample_percentile': percentiles}))
    print('Percentiles describe this dataset, not grades or recommended targets.')
    fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
    for swing in d.session_swing:
        m = motion_cache[swing]
        contact, (q, lead_z, total_z, lead_x) = trace_cache[swing]
        take = (m['time'] >= contact-PRIMARY_WINDOW_S) & (m['time'] <= contact-END_BEFORE_CONTACT_S)
        ms = 1000*(m['time'][take]-contact)
        axes[0].plot(ms, m['angles'][take, 0], alpha=.5)
        com = m['centerofmass'][take]
        axes[1].plot(ms, 100*(com[:, 0]-com[0, 0]), alpha=.5)
        axes[2].plot(1000*(q-contact), lead_z, alpha=.5)
    axes[0].set(xlabel='Time vs contact (ms)', ylabel='Lead-knee flexion (deg)', title='Knee movement across swings')
    axes[1].set(xlabel='Time vs contact (ms)', ylabel='Modeled COM forward movement (cm)', title='COM movement across swings')
    axes[2].set(xlabel='Time vs contact (ms)', ylabel='Measured lead vertical force (BW)', title='Measured force for interpretation')
    fig.suptitle('Hitter ' + hitter_id + ' — assessment profile')
    fig.tight_layout()
    fig.savefig(FIG / ('profile_' + hitter_id + '.png'), dpi=160, bbox_inches='tight')
    plt.show()
    return hitter_id

SELECTED_HITTER = None
selected_hitter = show_hitter(SELECTED_HITTER)


In [ ]:
AUDIT_NEAR_ZERO_BW = .10
AUDIT_LOW_FORCE_BW = .50
AUDIT_EVENT_TOLERANCE_BW = .30

def safe_value_at(t, values, when):
    if not np.isfinite(when) or when < t[0] or when > t[-1]:
        return np.nan
    j = int(np.searchsorted(t, when))
    if j == 0:
        return float(values[0]) if np.isfinite(values[0]) else np.nan
    if j == len(t):
        return float(values[-1]) if np.isfinite(values[-1]) else np.nan
    if not np.isfinite(values[j-1:j+1]).all():
        return np.nan
    return float(np.interp(when, t[j-1:j+1], values[j-1:j+1]))

def optional_event(frame, name):
    try:
        return scalar_event(frame, name)
    except ValueError:
        return np.nan

def audit_retained_trial(row, force_frame, angle_frame, landmark_frame, event_row):
    swing = row.session_swing
    t = force_frame.time.to_numpy(float)
    contact = scalar_event(force_frame, 'contact_time')
    bw = row.mass_kg * G
    lead = force_frame.lead_force_z.to_numpy(float)/bw
    rear = force_frame.rear_force_z.to_numpy(float)/bw
    total = lead + rear
    all_finite = np.isfinite(lead)
    peak_full = float(lead[all_finite].max()) if all_finite.any() else np.nan
    r = {'session_swing': swing, 'hitter_id': row.hitter_id, 'session_id': row.session_id,
         'contact_time': contact, 'lead_peak_window_bw': row.lead_peak_vertical_bw,
         'lead_peak_full_recording_bw': peak_full, 'total_peak_window_bw': row.total_peak_vertical_bw,
         'near_zero_lead_window': bool(row.lead_peak_vertical_bw < AUDIT_NEAR_ZERO_BW),
         'low_lead_window': bool(row.lead_peak_vertical_bw < AUDIT_LOW_FORCE_BW),
         'near_zero_lead_full_recording': bool(peak_full < AUDIT_NEAR_ZERO_BW),
         'force_peak_at_boundary': bool(row.force_peak_at_boundary),
         'kinematic_settling_vs_contact_ms': 1000*(event_row.kinematic_landing_time-contact),
         'early_total_bw_median': float(np.nanmedian(total[t <= t[0]+.100]))}
    ev = {}
    for name in EVENTS:
        values = [optional_event(table, name) for table in [force_frame, angle_frame, landmark_frame]]
        finite = np.asarray(values)[np.isfinite(values)]
        r[name + '_table_spread_ms'] = 1000*float(np.ptp(finite)) if len(finite)>1 else np.nan
        r[name + '_missing_tables'] = int(np.sum(~np.isfinite(values)))
        ev[name] = values[0]
    fc, fp = ev['fp_10_time'], ev['fp_100_time']
    r['official_contact_vs_contact_ms'] = 1000*(fc-contact)
    r['official_plant_vs_contact_ms'] = 1000*(fp-contact)
    r['official_events_missing'] = bool(not np.isfinite([fc, fp]).all())
    r['postcontact_landing_annotation'] = bool((np.isfinite(fc) and fc >= contact) or (np.isfinite(fp) and fp >= contact))
    r['reversed_contact_plant_annotation'] = bool(np.isfinite([fc, fp]).all() and fc >= fp)
    r['chronology_flag'] = r['postcontact_landing_annotation'] or r['reversed_contact_plant_annotation']
    r['lead_bw_at_official_contact'] = safe_value_at(t, lead, fc)
    r['lead_bw_at_official_plant'] = safe_value_at(t, lead, fp)
    r['contact_threshold_disagreement'] = bool(np.isfinite(r['lead_bw_at_official_contact']) and abs(r['lead_bw_at_official_contact']-.10) > AUDIT_EVENT_TOLERANCE_BW)
    r['plant_threshold_disagreement'] = bool(np.isfinite(r['lead_bw_at_official_plant']) and abs(r['lead_bw_at_official_plant']-1.) > AUDIT_EVENT_TOLERANCE_BW)
    win = (t >= contact-PRIMARY_WINDOW_S) & (t <= contact-END_BEFORE_CONTACT_S)
    r['window_lead_samples_above_10pct'] = float(np.mean(lead[win] >= .10))
    loaded = np.flatnonzero(np.isfinite(lead) & (lead >= .10))
    r['first_recorded_10pct_loading_vs_contact_ms'] = 1000*(t[loaded[0]]-contact) if len(loaded) else np.nan
    reasons = []
    if r['near_zero_lead_full_recording']:
        reasons.append('lead force stays near zero throughout recording; plate capture or assignment requires review')
    elif r['near_zero_lead_window']:
        reasons.append('near-zero force in primary window, with substantial loading elsewhere')
    elif r['low_lead_window']:
        reasons.append('low force in primary window')
    if r['chronology_flag']:
        reasons.append('official landing/plant chronology conflicts with contact or event order')
    if r['official_events_missing']:
        reasons.append('official landing or plant annotation missing')
    if r['contact_threshold_disagreement'] or r['plant_threshold_disagreement']:
        reasons.append('force level at annotated event differs from stated threshold')
    if any(r[e+'_table_spread_ms'] > .01 for e in EVENTS):
        reasons.append('event disagreement between source tables')
    r['review_reasons'] = '; '.join(reasons) if reasons else 'no selected review flags'
    r['screen_near_zero_or_chronology'] = r['near_zero_lead_window'] or r['chronology_flag']
    return r

event_index = events.set_index('session_swing')
force_audit = pd.DataFrame([audit_retained_trial(r, forces[r.session_swing], angles[r.session_swing],
                           landmarks[r.session_swing], event_index.loc[r.session_swing]) for _, r in primary.iterrows()])
assert len(force_audit) == len(primary) and not force_audit.session_swing.duplicated().any()
lead_oof = predictions[predictions.target.eq('lead_peak_vertical_bw') & predictions.model.eq('lower_body_com')]
lead_oof = lead_oof.groupby('session_swing')[['actual', 'predicted']].mean()
force_audit['repeat_averaged_lead_estimate_bw'] = force_audit.session_swing.map(lead_oof.predicted)
force_audit['lead_absolute_residual_bw'] = np.abs(force_audit.repeat_averaged_lead_estimate_bw-force_audit.lead_peak_window_bw)
force_audit.to_csv(OUT / 'retained_trial_force_audit.csv', index=False)
flags = ['near_zero_lead_window', 'low_lead_window', 'near_zero_lead_full_recording', 'chronology_flag',
         'official_events_missing', 'contact_threshold_disagreement', 'plant_threshold_disagreement',
         'screen_near_zero_or_chronology']
flag_summary = pd.DataFrame([{'flag': name, 'swings': int(force_audit[name].sum()),
                             'hitters': int(force_audit.loc[force_audit[name], 'hitter_id'].nunique())} for name in flags])
flag_summary.to_csv(OUT / 'force_audit_flag_counts.csv', index=False)
print('Measurement audit: retained primary swings only; prior exclusions remain in eligibility_audit.csv.')
print('Thresholds are exploratory review rules, not validated plate-capture criteria. No timestamps are repaired.')
print('Source threshold comparisons are diagnostic: sampling/filtering conventions can cause differences.')
display(flag_summary)
display(force_audit[force_audit.near_zero_lead_window][['session_swing', 'hitter_id', 'lead_peak_window_bw',
        'lead_peak_full_recording_bw', 'official_contact_vs_contact_ms', 'official_plant_vs_contact_ms', 'review_reasons']])

focus_ids = force_audit[force_audit.near_zero_lead_window].session_swing.tolist()
focus_ids += force_audit[force_audit.chronology_flag].sort_values('lead_absolute_residual_bw', ascending=False).session_swing.head(2).tolist()
focus_ids += force_audit.sort_values('lead_absolute_residual_bw', ascending=False).session_swing.head(2).tolist()
focus_ids = list(dict.fromkeys(focus_ids))[:6]
for swing in focus_ids:
    r = primary.loc[primary.session_swing.eq(swing)].iloc[0]
    f = forces[swing]
    m = motion_cache[swing]
    e = force_audit.loc[force_audit.session_swing.eq(swing)].iloc[0]
    contact = e.contact_time
    t = f.time.to_numpy(float)
    take = (t >= contact-.45) & (t <= contact+.30)
    ms = 1000*(t[take]-contact)
    lead_z = f.lead_force_z.to_numpy(float)/(r.mass_kg*G)
    rear_z = f.rear_force_z.to_numpy(float)/(r.mass_kg*G)
    fig, axes = plt.subplots(2, 1, figsize=(10, 6.5), sharex=True)
    axes[0].plot(ms, lead_z[take], label='Lead plate')
    axes[0].plot(ms, rear_z[take], label='Rear plate')
    axes[0].plot(ms, (lead_z+rear_z)[take], label='Sum', color='#555555', alpha=.65)
    axes[0].set(ylabel='Vertical force (BW)', title='Full source traces around recorded contact')
    for xval, label, color in [(e.official_contact_vs_contact_ms, 'Official 10% BW', '#d17a39'),
                               (e.official_plant_vs_contact_ms, 'Official 100% BW', '#9856a8')]:
        if np.isfinite(xval) and -450 <= xval <= 300:
            axes[0].axvline(xval, label=label, color=color, linestyle=':')
    lm = landmarks[swing]
    ankle_col = 'lajc_z' if r.hitter_side == 'R' else 'rajc_z'
    lt = lm.time.to_numpy(float)
    selected = (lt >= contact-.45) & (lt <= contact+.30)
    axes[1].plot(1000*(lt[selected]-contact), 100*lm[ankle_col].to_numpy(float)[selected], color='#167d9a')
    axes[1].set(xlabel='Time vs recorded bat contact (ms)', ylabel='Lead ankle height (cm)', title='Published ankle trajectory; not direct foot-ground contact')
    for ax in axes:
        ax.axvline(0, color='black', linestyle='--', linewidth=1)
        ax.axvspan(-150, -10, color='#167d9a', alpha=.10)
    axes[0].legend(fontsize=8, ncol=3)
    fig.suptitle('Trial '+swing+' | hitter '+r.hitter_id+' | shaded = unchanged primary window')
    fig.tight_layout()
    fig.savefig(FIG / ('audit_trial_'+swing+'.png'), dpi=150, bbox_inches='tight')
    plt.show()
print('Post-contact traces above are for measurement review only and never enter predictive features.')


In [ ]:
physics_table = primary.merge(physics[['session_swing', 'approx_total_peak_bw']], on='session_swing', validate='one_to_one')
physics_table = physics_table.rename(columns={'approx_total_peak_bw': 'physics_peak_bw'})
assert len(physics_table) == len(primary) and np.isfinite(physics_table.physics_peak_bw).all()
physics_sets = {'com_only_ridge': CONTROLS + COM_FEATURES, 'calibrated_com_physics': ['physics_peak_bw']}
physics_added, physics_tuning, physics_folds = nested_predictions(
    physics_table, ['total_peak_vertical_bw'], physics_sets, run_name='same_fold_total_force', outer_assignment=folds)
comparison = predictions[predictions.target.eq('total_peak_vertical_bw')].copy()
comparison['run'] = 'same_fold_total_force'
unfitted = comparison[comparison.model.eq('controls')].copy()
unfitted['model'] = 'unfitted_com_physics'
unfitted['alpha'] = np.nan
unfitted['predicted'] = unfitted.session_swing.map(physics_table.set_index('session_swing').physics_peak_bw)
physics_comparison = pd.concat([comparison, physics_added, unfitted], ignore_index=True)
counts = physics_comparison.groupby(['model', 'repeat']).session_swing.nunique()
assert counts.eq(len(primary)).all()
check_folds = physics_comparison.groupby(['session_swing', 'repeat']).fold.nunique()
assert check_folds.eq(1).all()
assert physics_comparison.groupby(['session_swing', 'repeat']).actual.nunique().eq(1).all()
assert physics_comparison.groupby(['model', 'repeat', 'session_swing']).size().eq(1).all()
physics_metrics = model_metrics(physics_comparison)
physics_pairs = pd.concat([
    paired_improvements(physics_comparison, 'lower_body_com', 'unfitted_com_physics'),
    paired_improvements(physics_comparison, 'lower_body_com', 'calibrated_com_physics'),
    paired_improvements(physics_comparison, 'lower_body_com', 'com_only_ridge'),
    paired_improvements(physics_comparison, 'calibrated_com_physics', 'unfitted_com_physics')], ignore_index=True)
for frame, filename in [(physics_comparison, 'same_fold_physics_predictions.csv'),
                         (physics_metrics, 'same_fold_physics_metrics.csv'),
                         (physics_pairs, 'same_fold_physics_bootstrap.csv'),
                         (physics_tuning, 'same_fold_physics_tuning.csv'), (physics_folds, 'same_fold_physics_folds.csv')]:
    frame.to_csv(OUT / filename, index=False)
print('Total-force comparison: identical trials, targets, repeat assignments and held-out hitter folds.')
print('Unfitted physics = max(1 + modeled COM vertical acceleration / g) within the primary window.')
print('Calibration is learned only in training folds. It corrects empirical scale/offset, not the underlying COM model.')
print('COM-only ridge uses COM features plus body controls. Full ridge additionally uses pelvis/joint motion.')
display(physics_metrics[['model', 'swings', 'hitters', 'mae', 'rmse', 'bias']])
display(physics_pairs)

order = ['mean', 'controls', 'lower_body', 'unfitted_com_physics', 'calibrated_com_physics', 'com_only_ridge', 'lower_body_com']
labels = ['Mean', 'Body controls', 'Lower body', 'COM physics', 'Calibrated physics', 'COM ridge', 'Lower body + COM']
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
mm = physics_metrics.set_index('model').reindex(order)
axes[0].barh(labels, mm.mae.to_numpy(), color=['#b7bdc6', '#8293a6', '#167d9a', '#b59b5a', '#bd8352', '#795a9d', '#d17a39'])
axes[0].invert_yaxis()
axes[0].set(xlabel='Hitter-weighted held-out MAE (BW)', title='Same-fold total-force comparison')
s = physics_comparison.assign(abs_error=lambda d: (d.predicted-d.actual).abs()).groupby(['hitter_id', 'model']).abs_error.mean().unstack()
axes[1].scatter(s.unfitted_com_physics, s.lower_body_com, alpha=.7, color='#167d9a', s=25)
upper = max(s.unfitted_com_physics.max(), s.lower_body_com.max())*1.05
axes[1].plot([0, upper], [0, upper], '--', color='#555555')
axes[1].set(xlabel='Unfitted COM physics: hitter MAE (BW)', ylabel='Lower body + COM: hitter MAE (BW)', title='One point per held-out hitter')
fig.tight_layout()
fig.savefig(FIG / '06_same_fold_physics_comparison.png', dpi=160, bbox_inches='tight')
plt.show()


In [ ]:
screen_definitions = {
    'omit_near_zero_lead': set(force_audit.loc[force_audit.near_zero_lead_window, 'session_swing']),
    'omit_near_zero_or_chronology': set(force_audit.loc[force_audit.screen_near_zero_or_chronology, 'session_swing'])
}
screen_prediction_frames, screen_count_rows, screen_comparison_rows = [], [], []
for name, excluded in screen_definitions.items():
    cohort = primary[~primary.session_swing.isin(excluded)].copy().reset_index(drop=True)
    screen_count_rows.append({'screen': name, 'excluded_swings': len(excluded), 'retained_swings': len(cohort),
                              'retained_hitters': cohort.hitter_id.nunique()})
    if not excluded:
        continue
    models = {k: MODEL_FEATURES[k] for k in ['controls', 'lower_body_com']}
    targets = ['lead_peak_vertical_bw', 'total_peak_vertical_bw']
    sp, st, sf = nested_predictions(cohort, targets, models, run_name=name+'_refit', outer_assignment=folds)
    screen_prediction_frames.append(sp)
    st.to_csv(OUT / (name+'_tuning.csv'), index=False)
    sf.to_csv(OUT / (name+'_folds.csv'), index=False)
    rescore = predictions[predictions.session_swing.isin(cohort.session_swing)
                         & predictions.target.isin(targets) & predictions.model.isin(models)].copy()
    rescore['run'] = name+'_original_model_rescored'
    screen_prediction_frames.append(rescore)
    keys = ['target', 'model', 'repeat', 'session_swing']
    matched = sp.merge(rescore, on=keys, suffixes=('_refit', '_original'), validate='one_to_one')
    assert len(matched) == len(sp) == len(rescore)
    assert matched.fold_refit.eq(matched.fold_original).all()
    assert np.allclose(matched.actual_refit, matched.actual_original)
    for target in targets:
        selected = matched[matched.target.eq(target) & matched.model.eq('lower_body_com')].copy()
        selected['delta'] = np.abs(selected.predicted_original-selected.actual_original)-np.abs(selected.predicted_refit-selected.actual_refit)
        d = selected.groupby('hitter_id_refit').delta.mean().to_numpy(float)
        rng = np.random.default_rng(SEED)
        boot = rng.choice(d, (BOOTSTRAPS, len(d)), replace=True).mean(axis=1)
        screen_comparison_rows.append({'screen': name, 'target': target, 'hitters': len(d),
                                      'mae_improvement_after_refit': d.mean(),
                                      'low': np.quantile(boot, .025), 'high': np.quantile(boot, .975)})
screen_counts = pd.DataFrame(screen_count_rows)
screen_predictions = pd.concat(screen_prediction_frames, ignore_index=True) if screen_prediction_frames else pd.DataFrame()
screen_metrics = model_metrics(screen_predictions) if len(screen_predictions) else pd.DataFrame()
screen_refit_comparison = pd.DataFrame(screen_comparison_rows)
for frame, filename in [(screen_counts, 'screen_cohort_counts.csv'), (screen_predictions, 'screen_predictions.csv'),
                         (screen_metrics, 'screen_model_metrics.csv'), (screen_refit_comparison, 'screen_refit_vs_rescore_bootstrap.csv')]:
    frame.to_csv(OUT / filename, index=False)
print('Exploratory exclusions do not replace the original primary results.')
print('Rescoring removes flagged test trials only. Refitting also removes them from training, preserving original outer hitter folds.')
print('Removing low-force targets changes the outcome distribution. Lower error on a screened cohort alone is not a model improvement.')
print('Flags cannot establish whether contact annotation, plate capture, or another source detail is wrong.')
display(screen_counts)
display(screen_metrics[['run', 'target', 'model', 'swings', 'hitters', 'mae']])
display(screen_refit_comparison)

def focused_audit_checks():
    assert np.isclose(safe_value_at(np.array([0., 1.]), np.array([0., 2.]), .25), .5)
    assert np.isnan(safe_value_at(np.array([0., 1.]), np.array([0., 2.]), 1.1))
    assert np.isnan(safe_value_at(np.array([0., 1.]), np.array([0., np.nan]), .5))
    trial = primary.iloc[0].copy()
    ft = np.arange(1081)/1080
    synthetic = pd.DataFrame({'time': ft, 'contact_time': .6, 'fp_10_time': .7, 'fp_100_time': .72,
                              'lead_force_z': 0., 'rear_force_z': trial.mass_kg*G})
    event = pd.Series({'kinematic_landing_time': .7})
    trial.lead_peak_vertical_bw = 0.
    trial.total_peak_vertical_bw = 1.
    out = audit_retained_trial(trial, synthetic, synthetic, synthetic, event)
    assert out['chronology_flag'] and out['near_zero_lead_full_recording']
    assert out['screen_near_zero_or_chronology']
    assert not np.isfinite(out['first_recorded_10pct_loading_vs_contact_ms'])
    y = np.array([1., 2., 3.])
    example = pd.DataFrame({'run': 'check', 'target': 'total_peak_vertical_bw', 'model': 'physics',
                            'hitter_id': ['a', 'a', 'b'], 'session_swing': ['a1','a2','b1'],
                            'repeat': 0, 'actual': y, 'predicted': y+.2})
    assert np.isclose(model_metrics(example).mae.iloc[0], .2)
    print('Audit checks passed: non-extrapolating event samples, missing signal handling, synthetic chronology/plate flags, paired folds and equal hitter scoring.')

focused_audit_checks()
AUDIT_SUMMARY = ['V2 FOCUSED MEASUREMENT AUDIT',
                 f'Retained-trial audit: {len(force_audit)} swings; near-zero lead window: {int(force_audit.near_zero_lead_window.sum())}; chronology flags: {int(force_audit.chronology_flag.sum())}.',
                 'Thresholds/omissions are exploratory diagnostics and do not replace the original primary analysis. No timestamps or force signals are repaired.',
                 'Near-zero lead force throughout the recording is consistent with incomplete lead-plate capture or assignment, but the released data cannot identify the cause.',
                 'Substantial loading outside a near-zero primary window plus conflicting events indicates an alignment concern, not proof of which timestamp is wrong.',
                 '', 'SAME-FOLD COM PHYSICS COMPARISON']
for _, row in physics_metrics.iterrows():
    AUDIT_SUMMARY.append(f'{row.model}: total-force peak MAE {row.mae:.4f} BW, RMSE {row.rmse:.4f} BW, bias {row.bias:+.4f} BW.')
for _, row in physics_pairs.iterrows():
    AUDIT_SUMMARY.append(f'{row.candidate} vs {row.reference}: paired MAE improvement {row.mae_improvement:+.4f} BW, conditional 95% interval [{row.low:+.4f}, {row.high:+.4f}].')
full_vs_calibrated = physics_pairs[(physics_pairs.candidate.eq('lower_body_com')) & physics_pairs.reference.eq('calibrated_com_physics')].iloc[0]
if full_vs_calibrated.low > 0:
    AUDIT_SUMMARY.append('The full model adds predictive information beyond calibrated COM physics in this dataset; external validation remains necessary.')
else:
    AUDIT_SUMMARY.append('Added predictive value of the full model beyond training-calibrated COM physics is not established. Improvements over body-size controls alone do not establish a need for the larger model.')
AUDIT_SUMMARY += ['', 'EXCLUSION SENSITIVITY']
for _, row in screen_counts.iterrows():
    AUDIT_SUMMARY.append(f'{row.screen}: excluded {row.excluded_swings} swings, retained {row.retained_swings} swings from {row.retained_hitters} hitters.')
AUDIT_SUMMARY += ['Screened errors answer a different cohort question. Refitted-versus-rescored comparisons use identical surviving trials and original outer folds.',
                  'Physics assumptions, COM modeling uncertainty, and lack of external validation still apply.']
if len(screen_refit_comparison) and not (screen_refit_comparison.low > 0).any():
    AUDIT_SUMMARY.append('None of these omission sensitivities clearly improved refitted-model error on the identical surviving trials.')
(OUT / 'focused_audit_interpretation.txt').write_text('\n'.join(AUDIT_SUMMARY), encoding='utf-8')


In [ ]:
LEAD_COMPARISON_TARGETS = ['lead_peak_vertical_bw', 'lead_braking_impulse_bws']
LEAD_COM_FEATURES = CONTROLS + COM_FEATURES
assert not set(LEAD_COM_FEATURES) & set(TARGETS + outcome_cols + MOTION_FEATURES)
lead_com_predictions, lead_com_tuning, lead_com_folds = nested_predictions(
    primary, LEAD_COMPARISON_TARGETS, {'com_only_ridge': LEAD_COM_FEATURES},
    run_name='same_fold_lead_loading', outer_assignment=folds)
lead_reference = predictions[predictions.target.isin(LEAD_COMPARISON_TARGETS)].copy()
lead_reference['run'] = 'same_fold_lead_loading'
lead_loading_predictions = pd.concat([lead_reference, lead_com_predictions], ignore_index=True)
expected_models = {'mean', 'controls', 'lower_body', 'lower_body_com', 'com_only_ridge'}
assert set(lead_loading_predictions.model) == expected_models
assert lead_loading_predictions.groupby(['target', 'model', 'repeat']).session_swing.nunique().eq(len(primary)).all()
assert lead_loading_predictions.groupby(['target', 'repeat', 'session_swing']).fold.nunique().eq(1).all()
assert lead_loading_predictions.groupby(['target', 'repeat', 'session_swing']).actual.nunique().eq(1).all()
assert lead_loading_predictions.groupby(['target', 'model', 'repeat', 'session_swing']).size().eq(1).all()
assert lead_loading_predictions.groupby(['target', 'repeat', 'hitter_id']).fold.nunique().eq(1).all()
lead_loading_metrics = model_metrics(lead_loading_predictions)
lead_loading_pairs = pd.concat([
    paired_improvements(lead_loading_predictions, 'lower_body_com', 'com_only_ridge'),
    paired_improvements(lead_loading_predictions, 'com_only_ridge', 'controls')], ignore_index=True)
repeat_rows = []
for (target, repeat), d in lead_loading_predictions.groupby(['target', 'repeat']):
    errors = d.assign(error=(d.predicted-d.actual).abs()).groupby(['hitter_id', 'model']).error.mean().unstack()
    repeat_rows.append({'target': target, 'repeat': repeat, 'hitters': len(errors),
                        'com_only_mae': errors.com_only_ridge.mean(), 'full_model_mae': errors.lower_body_com.mean(),
                        'full_over_com_mae_improvement': (errors.com_only_ridge-errors.lower_body_com).mean()})
lead_loading_repeats = pd.DataFrame(repeat_rows)
lead_screen_rows, lead_screen_pairs = [], []
for screen_name, excluded in screen_definitions.items():
    d = lead_loading_predictions[~lead_loading_predictions.session_swing.isin(excluded)].copy()
    d['run'] = 'lead_loading_rescore_' + screen_name
    lead_screen_rows.append(model_metrics(d))
    lead_screen_pairs.append(paired_improvements(d, 'lower_body_com', 'com_only_ridge'))
lead_loading_screen_metrics = pd.concat(lead_screen_rows, ignore_index=True)
lead_loading_screen_pairs = pd.concat(lead_screen_pairs, ignore_index=True)
for frame, filename in [(lead_loading_predictions, 'lead_loading_com_predictions.csv'),
                         (lead_com_tuning, 'lead_loading_com_tuning.csv'),
                         (lead_com_folds, 'lead_loading_com_folds.csv'),
                         (lead_loading_metrics, 'lead_loading_com_metrics.csv'),
                         (lead_loading_pairs, 'lead_loading_com_bootstrap.csv'),
                         (lead_loading_repeats, 'lead_loading_com_repeat_metrics.csv'),
                         (lead_loading_screen_metrics, 'lead_loading_screen_rescore_metrics.csv'),
                         (lead_loading_screen_pairs, 'lead_loading_screen_rescore_bootstrap.csv')]:
    frame.to_csv(OUT / filename, index=False)
print('Focused comparison: do pelvis/joint features add information beyond COM features for lead-side loading?')
print('COM-only = body controls + modeled COM features; full = the same features plus pelvis/joint movement.')
print('These are learned regressions. Total COM acceleration alone cannot allocate force to an individual foot.')
print('Both outcomes use exactly the same primary swings, outer hitter folds, inner split seeds, tuning grid and hitter weights.')
print('Positive improvement means the full model has lower error than COM-only. Bootstrap intervals condition on the trained predictions.')
display(lead_loading_metrics[['target', 'model', 'swings', 'hitters', 'mae', 'rmse', 'bias']])
display(lead_loading_pairs)
display(lead_loading_repeats)
print('Flag-screen comparisons below only rescore existing predictions on shared surviving trials; no model is refit or selected by these screens.')
display(lead_loading_screen_pairs)

lead_model_order = ['controls', 'lower_body', 'com_only_ridge', 'lower_body_com']
lead_labels = ['Body controls', 'Lower body', 'COM + controls', 'Lower body + COM']
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for col, target in enumerate(LEAD_COMPARISON_TARGETS):
    target_metrics = lead_loading_metrics[lead_loading_metrics.target.eq(target)].set_index('model').reindex(lead_model_order)
    unit = 'BW' if target == 'lead_peak_vertical_bw' else 'BW·s'
    axes[0, col].bar(lead_labels, target_metrics.mae.to_numpy(), color=['#8293a6', '#167d9a', '#795a9d', '#d17a39'])
    axes[0, col].tick_params(axis='x', rotation=18)
    axes[0, col].set(title=TARGET_LABELS[target], ylabel='Hitter-weighted MAE (' + unit + ')')
    d = lead_loading_predictions[lead_loading_predictions.target.eq(target)].copy()
    d['abs_error'] = np.abs(d.predicted-d.actual)
    errors = d.groupby(['hitter_id', 'model']).abs_error.mean().unstack()
    axes[1, col].scatter(errors.com_only_ridge, errors.lower_body_com, alpha=.7, s=25, color='#167d9a')
    upper = max(errors.com_only_ridge.max(), errors.lower_body_com.max())*1.05
    axes[1, col].plot([0, upper], [0, upper], '--', color='#555555')
    axes[1, col].set(xlabel='COM-only: hitter MAE ('+unit+')', ylabel='Full model: hitter MAE ('+unit+')',
                     title='One point per hitter; below diagonal favors full model')
fig.suptitle('Lead-side loading: matched COM-only versus full-model comparison', fontsize=14)
fig.tight_layout()
fig.savefig(FIG / '07_lead_loading_com_comparison.png', dpi=160, bbox_inches='tight')
plt.show()

LEAD_COMPARISON_SUMMARY = ['V3 FOCUSED LEAD-LOADING COMPARISON',
                          'Question: does adding lower-body movement to COM features improve prediction of lead vertical-force peak and braking impulse?',
                          'COM-only and full models share body controls, preprocessing/tuning procedures, primary trials and hitter-disjoint folds.']
for target in LEAD_COMPARISON_TARGETS:
    m = lead_loading_metrics[lead_loading_metrics.target.eq(target)].set_index('model')
    p = lead_loading_pairs[(lead_loading_pairs.target.eq(target)) & lead_loading_pairs.reference.eq('com_only_ridge')].iloc[0]
    verdict = 'added predictive information in this dataset' if p.low > 0 else 'no clear added predictive information' if p.high >= 0 else 'full model predicted worse in this dataset'
    LEAD_COMPARISON_SUMMARY.append(f'{TARGET_LABELS[target]}: COM-only MAE {m.loc["com_only_ridge", "mae"]:.6f}; full-model MAE {m.loc["lower_body_com", "mae"]:.6f}. Paired improvement {p.mae_improvement:+.6f}, conditional 95% interval [{p.low:+.6f}, {p.high:+.6f}]: {verdict}.')
LEAD_COMPARISON_SUMMARY += ['This comparison evaluates the lower-body feature set collectively; it does not identify a causal action or an ideal movement strategy.',
                           'A gain in lead-force prediction alone does not demonstrate accurate instantaneous force sharing, a full force waveform, or rotational torque.',
                           'Two outcomes and the screen checks are exploratory. Intervals are not adjusted for multiple comparisons or full retraining uncertainty.',
                           'Screened comparisons rescore fixed predictions on a different outcome distribution; the full primary cohort remains the main analysis.',
                           'These are assessments using processed optical motion capture and retrospective contact alignment; external and video-based validity remain untested.']
(OUT / 'lead_loading_com_interpretation.txt').write_text('\n'.join(LEAD_COMPARISON_SUMMARY), encoding='utf-8')
print('Comparison checks passed: identical trials/targets/folds, one prediction per swing/model/repeat, and hitter-disjoint held-out assignments.')


In [ ]:
from html import escape

def one_result(table, target, model):
    rows = table[table.target.eq(target) & table.model.eq(model)]
    if len(rows) != 1:
        raise ValueError(f'Expected one result for {target}/{model}; got {len(rows)}.')
    return rows.iloc[0]

def one_comparison(table, target, reference):
    rows = table[table.target.eq(target) & table.candidate.eq('lower_body_com') & table.reference.eq(reference)]
    if len(rows) != 1:
        raise ValueError(f'Expected one paired comparison for {target}/{reference}.')
    return rows.iloc[0]

closing_rows = []
closing_specs = [
    ('Total vertical-force peak', 'total_peak_vertical_bw', 'BW', physics_metrics, physics_pairs,
     'calibrated_com_physics', 'Calibrated COM physics', 4),
    ('Lead vertical-force peak', 'lead_peak_vertical_bw', 'BW', lead_loading_metrics, lead_loading_pairs,
     'com_only_ridge', 'COM features + controls', 4),
    ('Lead braking impulse', 'lead_braking_impulse_bws', 'BW·s', lead_loading_metrics, lead_loading_pairs,
     'com_only_ridge', 'COM features + controls', 5)
]
for label, target, unit, table, paired, reference, reference_label, digits in closing_specs:
    simple = one_result(table, target, reference)
    full = one_result(table, target, 'lower_body_com')
    comparison = one_comparison(paired, target, reference)
    if comparison.low > 0:
        assessment = 'Full model adds predictive information in this sample'
    elif comparison.high < 0:
        assessment = 'Simpler model has lower error in this sample'
    else:
        assessment = 'No clear added value from the full model'
    closing_rows.append({'Outcome': label, 'Unit': unit, 'Simpler benchmark': reference_label,
                         'Benchmark MAE': float(simple.mae), 'Full-model MAE': float(full.mae),
                         'Full-model improvement': float(comparison.mae_improvement),
                         '95% interval low': float(comparison.low), '95% interval high': float(comparison.high),
                         'Assessment': assessment, 'Digits': digits})
closing_results = pd.DataFrame(closing_rows)
closing_results.to_csv(OUT / 'study_summary_results.csv', index=False)
summary_table_rows = []
for _, row in closing_results.iterrows():
    digits = int(row.Digits)
    summary_table_rows.append({'Outcome': row.Outcome + ' (' + row.Unit + ')',
                               'Simpler benchmark': row['Simpler benchmark'],
                               'Benchmark MAE': f'{row["Benchmark MAE"]:.{digits}f}',
                               'Full-model MAE': f'{row["Full-model MAE"]:.{digits}f}',
                               'MAE improvement [95% interval]': f'{row["Full-model improvement"]:+.{digits}f} [{row["95% interval low"]:+.{digits}f}, {row["95% interval high"]:+.{digits}f}]',
                               'Reading': row.Assessment})
summary_table = pd.DataFrame(summary_table_rows)
summary_table_html = summary_table.to_html(index=False, border=0, classes='results', escape=True)

n_swings, n_hitters = len(primary), int(primary.hitter_id.nunique())
any_clear_gain = bool((closing_results['95% interval low'] > 0).any())
if any_clear_gain:
    takeaway = 'The value of additional lower-body features depends on the force measure. Read each paired comparison separately; a gain for one outcome does not establish a general advantage for the full model.'
else:
    takeaway = 'Modeled COM movement was informative about selected pre-contact force measures. Adding pelvis and joint features did not establish a reliable advantage across the tested outcomes. The simpler COM-based models are useful benchmarks for this dataset.'
timing_full = one_result(metrics, 'lead_peak_time_ms', 'lower_body_com')
timing_mean = one_result(metrics, 'lead_peak_time_ms', 'mean')
timing_pair = one_comparison(improvements, 'lead_peak_time_ms', 'controls')
timing_text = (f'Peak-force timing remained weak: full-model MAE was {timing_full.mae:.2f} ms, '
               f'versus {timing_mean.mae:.2f} ms for the training-mean baseline. '
               f'The paired improvement over body controls was {timing_pair.mae_improvement:+.2f} ms '
               f'(conditional 95% interval {timing_pair.low:+.2f} to {timing_pair.high:+.2f} ms).')
cluster_text = (f'{len(profiles)} hitter profiles formed {chosen_k} descriptive clusters. '
                f'The silhouette score was {silhouette:.3f} and median resampling ARI was {median_ari:.3f}. '
                'Stable assignments under swing resampling can coexist with overlapping groups; these profiles do not establish distinct hitter types or ideal mechanics.')
near_zero_n = int(force_audit.near_zero_lead_window.sum())
chronology_n = int(force_audit.chronology_flag.sum())
combined_screen = screen_counts[screen_counts.screen.eq('omit_near_zero_or_chronology')].iloc[0]
audit_text = (f'The retained-trial audit flagged {near_zero_n} near-zero lead-force windows and {chronology_n} landing/plant chronology concerns. '
              f'The combined omission sensitivity removed {int(combined_screen.excluded_swings)} unique swings, retaining '
              f'{int(combined_screen.retained_swings)} swings from {int(combined_screen.retained_hitters)} hitters. '
              'These are review flags, not proven recording errors. No timestamps were repaired and the full cohort remains primary.')
refit_text = ('None of the tested omissions clearly improved refitted-model error on identical surviving trials.'
              if not (screen_refit_comparison.low > 0).any()
              else 'Refitting effects differed by omission and outcome; inspect screen_refit_vs_rescore_bootstrap.csv before attributing a change to data cleaning.')
method_text = (f'The primary cohort contained {n_swings} swings from {n_hitters} hitters. '
               f'All force targets describe contact minus {PRIMARY_WINDOW_S*1000:.0f} ms through contact minus {END_BEFORE_CONTACT_S*1000:.0f} ms. '
               f'Validation used {OUTER_FOLDS} outer folds, {INNER_FOLDS} inner folds and {CV_REPEATS} repeated hitter splits. '
               'Whole hitters were held out, scaling and tuning used training data only, and fitting/scoring weighted hitters equally. '
               'Force and motion signals were evaluated on their own sampling grids. Force-derived landing timestamps were confined to the assisted benchmark and audit.')
model_note = ('COM features + controls is a ridge model using the published modeled COM trajectory, height, mass and handedness. '
              'The full model adds pelvis translation, hip/knee motion and stance geometry. '
              'Calibrated COM physics is a training-only regression of total-force peak on the approximate COM-acceleration peak. '
              'The comparisons use the same observed targets, trials and outer hitter folds. Calibration does not validate the underlying COM model.')
limits = [
    'This is retrospective assessment of processed optical motion capture. Contact alignment and upstream filtering are part of the measurement process; real-time or phone-video performance has not been tested.',
    'The published COM trajectory is model-derived. Segment definitions, bat mass, filtering and tracking error affect its relationship with measured ground reaction force.',
    'The results concern window-specific peaks and impulse, not full force waveforms, muscle force, rotational power, or an exact allocation of force between feet.',
    'Bootstrap intervals resample fixed per-hitter validation errors. They omit full retraining uncertainty, are not adjusted for multiple comparisons, and come from repeated exploratory work on one dataset.',
    'Screened rescoring and refitting answer different questions. Removing unusual targets changes the evaluation distribution and does not itself establish a better model.',
    'No external cohort or coaching intervention was tested. The models have not demonstrated that changing a movement feature improves performance, and this sample is not a normative grading system.'
]
practical_text = ('The study supports testing motion-derived force proxies against measured forces and keeping simple COM-based benchmarks in that evaluation. '
                  'It does not establish a replacement for force plates. Continuous hitter profiles can guide questions for further assessment; they are not coaching prescriptions.')
source_url = 'https://github.com/drivelineresearch/openbiomechanics/tree/dataset-v1/baseball_hitting'
license_url = 'https://github.com/drivelineresearch/openbiomechanics/blob/main/LICENSE-DATA.md'

report_style = '''
.study-summary {max-width:1200px;margin:18px auto;padding:28px;color:#17232d;background:#fff;border:1px solid #d9e1e7;border-radius:10px;font:15px/1.6 system-ui,sans-serif;}
.study-summary h2 {font-size:26px;line-height:1.25;margin:0 0 12px;color:#12364a;}
.study-summary h3 {font-size:18px;margin:24px 0 8px;color:#12364a;}
.study-summary p {margin:10px 0;}
.study-summary .takeaway {border-left:4px solid #167d9a;background:#eff6f8;padding:12px 16px;}
.study-summary .table-scroll {overflow-x:auto;margin:16px 0;}
.study-summary table {width:100%;border-collapse:collapse;font-size:13px;}
.study-summary th,.study-summary td {text-align:left;padding:11px 9px;border-bottom:1px solid #d9e1e7;vertical-align:top;}
.study-summary th {background:#eef3f6;}
.study-summary td:nth-child(3),.study-summary td:nth-child(4),.study-summary td:nth-child(5) {font-variant-numeric:tabular-nums;white-space:nowrap;}
.study-summary .note {font-size:13px;color:#485d6b;}
.study-summary li {margin:7px 0;}
.study-summary a {color:#116b8c;}
'''
summary_html = (f'<section class="study-summary"><h2>What the study found</h2>'
                f'<p class="takeaway">{escape(takeaway)}</p>'
                f'<p>{escape(method_text)}</p><h3>Matched model comparisons</h3>'
                f'<div class="table-scroll">{summary_table_html}</div>'
                '<p class="note">MAE is hitter-weighted mean absolute error; lower is better. BW = bodyweight; BW·s = bodyweight-seconds. '
                'Improvement is benchmark MAE minus full-model MAE, so positive values favor the full model. '
                'The intervals are conditional hitter-bootstrap intervals. Each row uses its stated unit.</p>'
                f'<p>{escape(model_note)}</p><h3>Timing and movement profiles</h3>'
                f'<p>{escape(timing_text)}</p><p>{escape(cluster_text)}</p>'
                f'<h3>Measurement audit</h3><p>{escape(audit_text)}</p><p>{escape(refit_text)}</p>'
                '<p class="note">The lead-loading comparisons kept the same interpretation after flagged trials were omitted from scoring. '
                'Those checks rescore existing predictions; they do not retrain the COM-only models.</p>'
                f'<h3>What this can support</h3><p>{escape(practical_text)}</p>'
                '<h3>Limits</h3><ul>' + ''.join('<li>'+escape(s)+'</li>' for s in limits) + '</ul>'
                '<h3>Reproduce the analysis</h3><p>Run all cells in order. Missing official inputs download automatically; input hashes, settings, exclusions, '
                'fold assignments, tuning results and predictions are exported with the run. Raw data are not embedded or redistributed.</p>'
                f'<p><a href="{source_url}">OpenBiomechanics Dataset v1 hitting data</a> · '
                f'<a href="{license_url}">Source data license</a></p></section>')
summary_document = '<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>Lower-body force proxies — study summary</title><style>'+report_style+'</style></head><body>'+summary_html+'</body></html>'
(OUT / 'study_summary.html').write_text(summary_document, encoding='utf-8')
summary_lines = ['LOWER-BODY MOVEMENT AND GROUND REACTION FORCE ESTIMATION', '', takeaway, '', method_text,
                 '', 'MATCHED MODEL COMPARISONS', summary_table.to_string(index=False),
                 'Positive MAE improvement favors the full model. Intervals are conditional hitter-bootstrap intervals; use the row-specific unit.',
                 '', model_note, '', 'TIMING AND MOVEMENT PROFILES', timing_text, cluster_text,
                 '', 'MEASUREMENT AUDIT', audit_text, refit_text,
                 'Lead-loading screen checks rescore existing predictions and do not retrain COM-only models.',
                 '', 'PRACTICAL INTERPRETATION', practical_text, '', 'LIMITS', *limits,
                 '', 'SOURCE', source_url, license_url]
interpretation = '\n'.join(summary_lines)
(OUT / 'interpretation.txt').write_text(interpretation, encoding='utf-8')
try:
    from IPython.display import HTML
    display(HTML('<style>'+report_style+'</style>'+summary_html))
except ImportError:
    print(interpretation)

definitions = [{'feature': c, 'group': 'modeled_com' if c in COM_FEATURES else 'lower_body',
                'meaning': 'Peak = maximum over analysis window; min = minimum; change = end minus start; knee/hip extension = negative flexion derivative; timing = ms relative to window end.'}
               for c in MOTION_FEATURES + COM_FEATURES]
pd.DataFrame(definitions).to_csv(OUT / 'feature_dictionary.csv', index=False)
import scipy, sklearn
manifest = {'revision': 3, 'presentation': 'final consolidated summary', 'lead_loading_comparison': {'targets': LEAD_COMPARISON_TARGETS, 'com_only_features': LEAD_COM_FEATURES}, 'focused_audit_thresholds': {'near_zero_bw': AUDIT_NEAR_ZERO_BW, 'low_force_bw': AUDIT_LOW_FORCE_BW, 'event_tolerance_bw': AUDIT_EVENT_TOLERANCE_BW}, 'study': 18, 'dataset': 'dataset-v1', 'input_sha256': INPUT_HASHES,
            'archive_expected_sha256': ARCHIVE_HASHES, 'seed': SEED,
            'settings': {'primary_window_s': PRIMARY_WINDOW_S, 'end_before_contact_s': END_BEFORE_CONTACT_S,
                         'filter_hz': FILTER_HZ, 'minimum_profile_swings': MIN_PROFILE_SWINGS,
                         'outer_folds': OUTER_FOLDS, 'inner_folds': INNER_FOLDS, 'cv_repeats': CV_REPEATS,
                         'alphas': ALPHAS, 'bootstrap_draws': BOOTSTRAPS, 'cluster_resamples': CLUSTER_BOOTSTRAPS,
                         'configs': CONFIGS, 'run_model_sensitivity': RUN_MODEL_SENSITIVITY},
            'primary_cohort': {'swings': len(primary), 'hitters': primary.hitter_id.nunique()},
            'model_features': MODEL_FEATURES, 'targets': TARGETS, 'cluster_features': CLUSTER_FEATURES,
            'cluster_k': chosen_k, 'selected_hitter': selected_hitter,
            'versions': {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
                         'scipy': scipy.__version__, 'sklearn': sklearn.__version__}}
(OUT / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print('\nResults:', OUT)
print('Figures:', FIG)
